In [2]:
import os
import pickle

import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras import layers
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import (
    ModelCheckpoint,
    EarlyStopping,
    ReduceLROnPlateau,
    CSVLogger
)

In [3]:
print("TensorFlow Version :", tf.__version__)
print("GPU Available :", tf.config.list_physical_devices("GPU"))

TensorFlow Version : 2.16.1
GPU Available : []


In [4]:
SEED = 42

tf.random.set_seed(SEED)

In [5]:
IMG_SIZE = (224, 224)

BATCH_SIZE = 32

INITIAL_EPOCHS = 25

FINE_TUNE_EPOCHS = 30

PHASE2_EPOCHS = 20

In [6]:
dataset_path = "../datasets/food-101/images"

print(dataset_path)

../datasets/food-101/images


In [7]:
print(os.path.exists(dataset_path))

False


In [8]:
import os

print(os.getcwd())

c:\Users\spavi\ICBT CAMPUS FOLDER\AI-Based-Food-Recognition-System\ai_model\notebooks


In [9]:
dataset_path = "../../datasets/food-101/images"

print(dataset_path)
print(os.path.exists(dataset_path))

../../datasets/food-101/images
True


In [10]:
train_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

Found 101000 files belonging to 101 classes.
Using 80800 files for training.


In [11]:
validation_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

Found 101000 files belonging to 101 classes.
Using 20200 files for validation.


In [12]:
NUM_CLASSES = len(train_dataset.class_names)

print("Number of Classes :", NUM_CLASSES)

Number of Classes : 101


In [13]:
AUTOTUNE = tf.data.AUTOTUNE

train_dataset = train_dataset.prefetch(AUTOTUNE)
validation_dataset = validation_dataset.prefetch(AUTOTUNE)

In [14]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2),
    layers.RandomContrast(0.2),
], name="data_augmentation")

In [15]:
preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input

In [16]:
base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

In [17]:
base_model.trainable = False

print("Trainable :", base_model.trainable)

Trainable : False


In [18]:
inputs = tf.keras.Input(shape=(224, 224, 3))

x = data_augmentation(inputs)

x = preprocess_input(x)

x = base_model(x, training=False)

x = layers.GlobalAveragePooling2D()(x)

x = layers.Dropout(0.3)(x)

outputs = layers.Dense(
    NUM_CLASSES,
    activation="softmax"
)(x)

model = Model(inputs, outputs)

In [19]:
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ data_augmentation (Sequential)  │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide (TrueDivide)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract (Subtract)             │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 101)            │       129,381 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,387,365 (9.11 MB)

 Trainable params: 129,381 (505.39 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [20]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("Model Compiled Successfully!")

Model Compiled Successfully!


In [21]:
os.makedirs("../saved_models", exist_ok=True)

print("Saved Models Folder Ready!")

Saved Models Folder Ready!


In [22]:
checkpoint = ModelCheckpoint(
    "../saved_models/mobilenetv2_initial.keras",
    monitor="val_accuracy",
    save_best_only=True,
    mode="max",
    verbose=1
)

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

csv_logger = CSVLogger(
    "../saved_models/training_log.csv"
)

In [25]:
history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=INITIAL_EPOCHS,
    callbacks=[
        checkpoint,
        early_stopping,
        reduce_lr,
        csv_logger
    ]
)

Epoch 1/25


2525/2525 ━━━━━━━━━━━━━━━━━━━━ 0s 607ms/step - accuracy: 0.3162 - loss: 2.9581
Epoch 1: val_accuracy improved from None to 0.53287, saving model to ../saved_models/mobilenetv2_initial.keras

Epoch 1: finished saving model to ../saved_models/mobilenetv2_initial.keras
2525/2525 ━━━━━━━━━━━━━━━━━━━━ 1878s 741ms/step - accuracy: 0.3948 - loss: 2.5188 - val_accuracy: 0.5329 - val_loss: 1.8365 - learning_rate: 0.0010
Epoch 2/25
2525/2525 ━━━━━━━━━━━━━━━━━━━━ 0s 651ms/step - accuracy: 0.4676 - loss: 2.1879
Epoch 2: val_accuracy improved from 0.53287 to 0.54460, saving model to ../saved_models/mobilenetv2_initial.keras

Epoch 2: finished saving model to ../saved_models/mobilenetv2_initial.keras
2525/2525 ━━━━━━━━━━━━━━━━━━━━ 2006s 794ms/step - accuracy: 0.4693 - loss: 2.1764 - val_accuracy: 0.5446 - val_loss: 1.7943 - learning_rate: 0.0010
Epoch 3/25
2525/2525 ━━━━━━━━━━━━━━━━━━━━ 0s 646ms/step - accuracy: 0.4787 - loss: 2.1437
Epoch 3: val_accuracy improved from 0.54460 to 0.54941, saving mod

In [23]:
from tensorflow.keras.models import load_model

model = load_model("../saved_models/mobilenetv2_initial.keras")

print("Initial Model Loaded")

Initial Model Loaded


In [24]:
for i, layer in enumerate(model.layers):
    print(i, layer.name, type(layer))

0 input_layer_1 <class 'keras.src.layers.core.input_layer.InputLayer'>
1 data_augmentation <class 'keras.src.models.sequential.Sequential'>
2 mobilenetv2_1.00_224 <class 'keras.src.models.functional.Functional'>
3 global_average_pooling2d <class 'keras.src.layers.pooling.global_average_pooling2d.GlobalAveragePooling2D'>
4 dropout <class 'keras.src.layers.regularization.dropout.Dropout'>
5 dense <class 'keras.src.layers.core.dense.Dense'>


In [25]:
base_model = model.layers[2]

print(base_model.name)
print(base_model.trainable)

mobilenetv2_1.00_224
False


In [26]:
base_model.trainable = True

for layer in base_model.layers[:-50]:
    layer.trainable = False

for layer in base_model.layers[-50:]:
    layer.trainable = True

print("Fine Tuning Ready")

Fine Tuning Ready


In [27]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [28]:
checkpoint_finetune = ModelCheckpoint(
    "../saved_models/mobilenetv2_finetuned.keras",
    monitor="val_accuracy",
    save_best_only=True,
    mode="max",
    verbose=1
)

early_stopping_finetune = EarlyStopping(
    monitor="val_loss",
    patience=7,
    restore_best_weights=True,
    verbose=1
)

reduce_lr_finetune = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=3,
    min_lr=1e-7,
    verbose=1
)

In [34]:
history_finetune = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=50,
    callbacks=[
        checkpoint_finetune,
        early_stopping_finetune,
        reduce_lr_finetune
    ]
)

Epoch 1/50
2525/2525 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.4001 - loss: 2.5203
Epoch 1: val_accuracy improved from None to 0.58921, saving model to ../saved_models/mobilenetv2_finetuned.keras

Epoch 1: finished saving model to ../saved_models/mobilenetv2_finetuned.keras
2525/2525 ━━━━━━━━━━━━━━━━━━━━ 3035s 1s/step - accuracy: 0.4569 - loss: 2.1884 - val_accuracy: 0.5892 - val_loss: 1.5978 - learning_rate: 1.0000e-05
Epoch 2/50
2525/2525 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.5321 - loss: 1.8347
Epoch 2: val_accuracy improved from 0.58921 to 0.62005, saving model to ../saved_models/mobilenetv2_finetuned.keras

Epoch 2: finished saving model to ../saved_models/mobilenetv2_finetuned.keras
2525/2525 ━━━━━━━━━━━━━━━━━━━━ 2957s 1s/step - accuracy: 0.5425 - loss: 1.7825 - val_accuracy: 0.6200 - val_loss: 1.4792 - learning_rate: 1.0000e-05
Epoch 3/50
2525/2525 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.5676 - loss: 1.6706
Epoch 3: val_accuracy improved from 0.62005 to 0.63797

In [35]:
import os

print(os.path.exists("../saved_models/mobilenetv2_finetuned.keras"))

True
